# Run Match

In [1]:
%load_ext autoreload
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100`% !important; }</style>"))
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=UserWarning)

In [2]:
from timeutils import Timestat
from master import MasterParams, MasterDBs, MasterPaths, MasterMetas, MusicDBPermDir
from musicdb import PanDBIO
from gate import MusicDBGate
from pandas import DataFrame, Series, concat
import musicdb
from ioutils import HTMLIO, FileIO
from listUtils import getFlatList
import swifter
import dask.dataframe as dd
from match import MatchDBDataIO, AlbumReq, NameReq, MatchReq, MatchDB, CrossMatchDB, PanDBMatch
io = FileIO()

In [21]:
baseReqs = {"MetalArchives": 4,
            "RateYourMusic": 20,
            "Beatport": 35,
            "Spotify": 20,
            "Discogs": 3,  ## 12
            "Traxsource": 100000}
#baseDB    = "Beatport"
#baseDB    = "Discogs"
#baseDB    = "Spotify"
#baseDB    = "Traxsource"
#baseDB    = "MyMixTapez"
#baseDB    = "Genius"
#baseDB    = "MetalArchives"
baseDB    = "RateYourMusic"  # 3

minL = 15
maxL = 150000

minA = 1
maxA = 30000000

#baseReq   = {baseDB: MatchReq(NameReq(min=minL, max=maxL), AlbumReq(min=baseReqs.get(baseDB), max=baseReqs.get(baseDB)+1))}
baseReq   = {baseDB: MatchReq(NameReq(min=minL, max=maxL), AlbumReq(min=minA, max=maxA))}
#baseReq   = {baseDB: AlbumReq(min=baseReqs.get(baseDB), max=baseReqs.get(baseDB)+1)}
#baseReq   = {baseDB: AlbumReq(min=10, max=baseReqs.get(baseDB,10000)+1, rnd=10000)}

compareDBs  = ["RateYourMusic", "LastFM", "Spotify", "Genius", "Discogs", "MusicBrainz", "Deezer", "MetalArchives"]
compareDBs  = ["RateYourMusic", "Spotify", "Genius", "Discogs", "MusicBrainz", "LastFM", "Deezer"]
compareDBs  = ["RateYourMusic", "Spotify", "Genius", "Discogs", "MusicBrainz", "Beatport"] # "LastFM", "Deezer"]
compareDBs  = ["RateYourMusic", "Spotify", "Discogs", "MusicBrainz", "Beatport", "Traxsource", "Genius", "MyMixTapez", "MetalArchives"] # "LastFM", "Deezer"]
#compareDBs  = ["RateYourMusic", "Traxsource", "Spotify", "Beatport"]
compareReqs = {compareDB: MatchReq(NameReq(min=minL-5, max=maxL+5), AlbumReq(min=3)) for compareDB in compareDBs if compareDB not in [baseDB]}
compareDBs  = list(compareReqs.keys())

matchReqs  = {**baseReq, **compareReqs}
mediaTypes = ["Album", "SingleEP"]
mediaTypes = ["{0}Media".format(mediaType) for mediaType in mediaTypes]
mediaTypes = list(MasterMetas().getMedias().values())
reqs       = {"Media": mediaTypes, "Reqs": matchReqs, "Mask": baseDB, "NPart": 3, "Match": {"Artist": 0.85, "Medium": 2, "Tight": 1}}
print("Primary Run Params:")
print("  ==> DBs:   [{0}] x {1}]".format(baseDB,list(compareReqs.keys())))
print("  ==> Media: {0}".format(mediaTypes))
print("  ==> Match: {0}".format(reqs["Match"]))

crossreqs  = {"Media": mediaTypes, "Reqs": {baseDB: MatchReq(AlbumReq(min=2))}, "Mask": baseDB, "NPart": 3, "Match": {"Artist": 0.85, "Medium": 2, "Tight": 1}}
print("Cross Match Run Params:")
print("  ==> DBs:   {0} x [{1}]".format(list(compareReqs.keys()), baseDB))
print("  ==> Media: {0}".format(mediaTypes))
print("  ==> Match: {0}".format(crossreqs["Match"]))

Primary Run Params:
  ==> DBs:   [RateYourMusic] x ['Spotify', 'Discogs', 'MusicBrainz', 'Beatport', 'Traxsource', 'Genius', 'MyMixTapez', 'MetalArchives']]
  ==> Media: ['Album', 'SingleEP', 'Appearance', 'Technical', 'Mix', 'Bootleg', 'AltMedia', 'Other']
  ==> Match: {'Artist': 0.85, 'Medium': 2, 'Tight': 1}
Cross Match Run Params:
  ==> DBs:   ['Spotify', 'Discogs', 'MusicBrainz', 'Beatport', 'Traxsource', 'Genius', 'MyMixTapez', 'MetalArchives'] x [RateYourMusic]
  ==> Media: ['Album', 'SingleEP', 'Appearance', 'Technical', 'Mix', 'Bootleg', 'AltMedia', 'Other']
  ==> Match: {'Artist': 0.85, 'Medium': 2, 'Tight': 1}


In [22]:
baseIO = MatchDBDataIO(db=baseDB, mediaTypes=reqs["Media"], mask=reqs["Mask"], verbose=True, base=True)
baseIO.loadNames()
baseIO.setAvailableNames(reqs["Reqs"][baseDB])
del baseIO

  MatchDBData(RateYourMusic):
   ==> Loading Names Data
     ==> Found   74452 Artists Basic Data In RateYourMusic DB
       ==> Masking Artists
     ==> Found   81750 Previously Cross Matched Artists
     ==> Found    8264 Artists In RateYourMusic DB
     ==> Found    8264 Available Artists In RateYourMusic DB
     ==> Cut[Media]             8264 Artists With [1/313] Min/Max
     ==> Cut[MediaLen]          8264 Artists With [1/313] Min/Max
     ==> Cut[Names]             8264 Artists With [1/102] Min/Max
     ==> Cut[NamesLen]          2597 Artists With [15/102] Min/Max
     ==> Cut[Final]             2597 Artists With [/] Min/Max


# Match & Cross Match

In [23]:
mdb = MatchDB(baseDB=baseDB, compareDBs=compareDBs, reqs=reqs)
mdb.match()
mdb.save()
del mdb

*******************************************************************************************************************************************************************************
*                                                                       MatchDB()                                                                                             *
*******************************************************************************************************************************************************************************
Process [Matching [RateYourMusic] Against ['Spotify', 'Discogs', 'MusicBrainz', 'Beatport', 'Traxsource', 'Genius', 'MyMixTapez', 'MetalArchives']] Start    ==> Time Is 2022-07-01 13:27:04

------------------------------------------------------------------------------------------------------------------------------------------------------
---------------------------------------------------------------------- RateYourMusic ----------------------------------------------

100%|██████████| 866/866 [04:58<00:00,  2.90it/s]


  Process [String Matching 2597 [RateYourMusic] x 284523 [Spotify] Artist Names] Ran For 5 Minutes and 1 Second    ==> Time Is 2022-07-01 13:32:20
     ==> Found 2597 Name Results
     ==> Found 329 Artists With One Or More Matches
     ==> Found 565 Possible Matches
    Process [Loading RateYourMusic Media Data] Start    ==> Time Is 2022-07-01 13:32:20
    Process [Loading RateYourMusic Media Data] Ran For 6 Seconds    ==> Time Is 2022-07-01 13:32:26
    Process [Loading Spotify Media Data] Start    ==> Time Is 2022-07-01 13:32:26
    Process [Loading Spotify Media Data] Ran For 20 Seconds    ==> Time Is 2022-07-01 13:32:46
  Process [String Matching 565 [RateYourMusic] Album Names] Start    ==> Time Is 2022-07-01 13:32:46
      Process [Matching 565 Artists' Albums] Start    ==> Time Is 2022-07-01 13:32:46
      Process [Matching 565 Artists' Albums] Ran For 3 Seconds    ==> Time Is 2022-07-01 13:32:49
By DB:
  Spotify             60
By NameQuality:
  Pure   47
  Great  2
  Good   3


100%|██████████| 866/866 [11:40<00:00,  1.24it/s]


  Process [String Matching 2597 [RateYourMusic] x 699244 [Discogs] Artist Names] Ran For 11 Minutes and 45 Seconds    ==> Time Is 2022-07-01 13:45:05
     ==> Found 2597 Name Results
     ==> Found 737 Artists With One Or More Matches
     ==> Found 1416 Possible Matches
    Process [Loading RateYourMusic Media Data] Start    ==> Time Is 2022-07-01 13:45:05
    Process [Loading RateYourMusic Media Data] Ran For     ==> Time Is 2022-07-01 13:45:05
    Process [Loading Discogs Media Data] Start    ==> Time Is 2022-07-01 13:45:05
    Process [Loading Discogs Media Data] Ran For 55 Seconds    ==> Time Is 2022-07-01 13:46:00
  Process [String Matching 1416 [RateYourMusic] Album Names] Start    ==> Time Is 2022-07-01 13:46:00
      Process [Matching 1416 Artists' Albums] Start    ==> Time Is 2022-07-01 13:46:00
      Process [Matching 1416 Artists' Albums] Ran For 4 Seconds    ==> Time Is 2022-07-01 13:46:04
By DB:
  Discogs             341
  Spotify             60
By NameQuality:
  Pure   2

...

  Process [String Matching 2597 [RateYourMusic] x 4135 [MyMixTapez] Artist Names] Ran For 6 Seconds    ==> Time Is 2022-07-01 13:52:43
     ==> Found 2597 Name Results
     ==> Found 2 Artists With One Or More Matches
     ==> Found 2 Possible Matches
    Process [Loading RateYourMusic Media Data] Start    ==> Time Is 2022-07-01 13:52:43
    Process [Loading RateYourMusic Media Data] Ran For     ==> Time Is 2022-07-01 13:52:43
    Process [Loading MyMixTapez Media Data] Start    ==> Time Is 2022-07-01 13:52:43
    Process [Loading MyMixTapez Media Data] Ran For     ==> Time Is 2022-07-01 13:52:44
  Process [String Matching 2 [RateYourMusic] Album Names] Start    ==> Time Is 2022-07-01 13:52:44
      Process [Matching 2 Artists' Albums] Start    ==> Time Is 2022-07-01 13:52:44


/Users/tgadfort/opt/anaconda3/envs/py310/lib/python3.9/site-packages/pandb-0.0.1-py3.9.egg/match/pool.py:24: FutureWarning: The default dtype for empty Series will be 'object' instead of 'float64' in a future version. Specify a dtype explicitly to silence this warning.
  retval = Series({key: poolMatchStringSeries(value["Base"], value["Compare"], showProgress=False, cutoff=kwargs.get('cutoff')) for key,value in item.items()})


      Process [Matching 2 Artists' Albums] Ran For 2 Seconds    ==> Time Is 2022-07-01 13:52:46
By DB:
  Discogs             341
  Spotify             60
  MusicBrainz         30
  Genius              30
  Traxsource          15
  Beatport            11
  MyMixTapez          1
By NameQuality:
  Pure   342
  Great  21
  Good   25
  Near   100
  Low    0
By MediaQuality:
Pure   181
Great  62
Good   7
Sole   199
Near   16
Loose  1
Low    2
Poor   20
  Process [String Matching 2 [RateYourMusic] Album Names] Ran For 2 Seconds    ==> Time Is 2022-07-01 13:52:46

------------------------------------------------------------------------------------------------------------------------------------------------------
---------------------------------------------------------------------- MetalArchives -----------------------------------------------------------------
  Process [Loading Artist Names] Start    ==> Time Is 2022-07-01 13:52:46
     ==> Cut[Media]            80367 Artists With [1/345] Min

100%|██████████| 866/866 [00:20<00:00, 41.47it/s]


  Process [String Matching 2597 [RateYourMusic] x 22352 [MetalArchives] Artist Names] Ran For 23 Seconds    ==> Time Is 2022-07-01 13:53:12
     ==> Found 2597 Name Results
     ==> Found 28 Artists With One Or More Matches
     ==> Found 30 Possible Matches
    Process [Loading RateYourMusic Media Data] Start    ==> Time Is 2022-07-01 13:53:12
    Process [Loading RateYourMusic Media Data] Ran For     ==> Time Is 2022-07-01 13:53:12
    Process [Loading MetalArchives Media Data] Start    ==> Time Is 2022-07-01 13:53:12
    Process [Loading MetalArchives Media Data] Ran For 1 Second    ==> Time Is 2022-07-01 13:53:13
  Process [String Matching 30 [RateYourMusic] Album Names] Start    ==> Time Is 2022-07-01 13:53:13
      Process [Matching 30 Artists' Albums] Start    ==> Time Is 2022-07-01 13:53:14
      Process [Matching 30 Artists' Albums] Ran For 2 Seconds    ==> Time Is 2022-07-01 13:53:15
By DB:
  Discogs             341
  Spotify             60
  MusicBrainz         30
  Genius  

In [24]:
mdbpd = MusicDBPermDir()
mres  = io.get(mdbpd.getMatchPermPath().join("primaryMatch.p"))
cmdb  = CrossMatchDB(baseDB, mres, crossreqs, verbose=True)
cmdb.match()
cmdb.save()

del cmdb

*******************************************************************************************************************************************************************************
*                                                                       MatchDB()                                                                                             *
*******************************************************************************************************************************************************************************
Process [Cross Matching [RateYourMusic] Against ['Spotify', 'Discogs', 'MusicBrainz', 'Beatport', 'Traxsource', 'Genius', 'MyMixTapez', 'MetalArchives']] Start    ==> Time Is 2022-07-01 13:53:16
     ==> Cut[Media]             8264 Artists With [1/313] Min/Max
     ==> Cut[MediaLen]          4613 Artists With [2/313] Min/Max
     ==> Cut[Final]             4613 Artists With [/] Min/Max
  Process [String Matching 501 ['Spotify', 'Discogs', 'MusicBrainz', 'Beatport', 'T

100%|██████████| 167/167 [00:00<00:00, 197.39it/s]


     ==> Found 501 Name Results
     ==> Found 260 Artists With One Or More Matches
     ==> Found 263 Possible Matches
  Process [String Matching 501 ['Spotify', 'Discogs', 'MusicBrainz', 'Beatport', 'Traxsource', 'Genius', 'MyMixTapez', 'MetalArchives'] x 4613 [RateYourMusic] Artist Names] Ran For 5 Seconds    ==> Time Is 2022-07-01 13:53:25
  Process [String Matching 263 ['Spotify', 'Discogs', 'MusicBrainz', 'Beatport', 'Traxsource', 'Genius', 'MyMixTapez', 'MetalArchives'] Album Names] Start    ==> Time Is 2022-07-01 13:53:25
      Process [Matching 263 Artists' Albums] Start    ==> Time Is 2022-07-01 13:53:25
      Process [Matching 263 Artists' Albums] Ran For 4 Seconds    ==> Time Is 2022-07-01 13:53:29
  Process [String Matching 263 ['Spotify', 'Discogs', 'MusicBrainz', 'Beatport', 'Traxsource', 'Genius', 'MyMixTapez', 'MetalArchives'] Album Names] Ran For 6 Seconds    ==> Time Is 2022-07-01 13:53:31
Saving Cross Match Results To /Users/tgadfort/opt/anaconda3/envs/py310/pandb/m

In [25]:
pdbm = PanDBMatch(baseDB, verbose=True)
pdbm.select(minQual=5, minName=5)
pdbm.master()
pdbm.merge(allownew=True, verbose=False)


*******************************************************************************************************************************************************************************
**                                                                             PanDBMatch()                                                                                  **
*******************************************************************************************************************************************************************************
6e67496fe2   | Discogs        319860                                    5   8    | THE GORDIAN KNOT                        THE GORDIAN KNOT
bcdbcf58e8   | Discogs        1282844                                   5   7    | JOHN BASSMAN GROUP                      JOHN BASSMAN GROUP
35de42d9d6   | Discogs        660565                                    5   5    | NANOOK OF THE NORTH                     NANOOK OF THE NORTH
c0cbee3755   | Beatport       2796            

cede5454ab   | Genius         454813                                    5   7    | VINCENT DE MOOR                         VINCENT DE MOOR
28b1fa11fe   | MusicBrainz    218698310583505117636498856505309116890   5   8    | VINCENT DE MOOR                         VINCENT DE MOOR
932ef2366c   | Spotify        513hutOhfryax7g1N0XHEk                    5   8    | VINCENT DE MOOR                         VINCENT DE MOOR
4e87fd9493   | Traxsource     47038                                     5   8    | VINCENT DE MOOR                         VINCENT DE MOOR
3e2c65a794   | Discogs        3564603                                   5   8    | BLACKWITCH PUDDING                      BLACKWITCH PUDDING
7e878fe605   | Discogs        3250147                                   5   5    | BLACK STAR MAFIA                        BLACK STAR MAFIA
24cc7ed235   | Spotify        1ZwmpNX1KvmgALKLT393VQ                    5   8    | BLACK STAR MAFIA                        BLACK STAR MAFIA
874e066b3a   | Discogs

In [26]:
pdbm.mergeMultiRows()

Nothing to merge


# Extra Match

In [ ]:
# 
# 2f60ed24b5   | Spotify        5NokRbYYfqacBmRVBRj0wD                    4   8    | THE VIOLENTS                            THE VIOLENT
# d747043fb9   | Discogs        415652                                    3   5    | TOM SMOTHERS                            TOMMY SMOTHERS

In [ ]:
pdbm = PanDBMatch(baseDB, verbose=True)
pdbm.select(minQual=5, minName=3, maxName=5)

In [ ]:
pdbm.include("""
6d358553dc   | Discogs        103279                                    4   8    | 60 FT DOLLS                             60FT DOLLS
9e73b5a98b   | Genius         993239                                    4   8    | 60 FT DOLLS                             60FT DOLLS
6784cd84dc   | MusicBrainz    45324672561978998016738221472695822933    4   8    | 60 FT DOLLS                             60FT DOLLS
0ecd4adf32   | Spotify        6XaT6lZurc7ceDovVXISOi                    4   7    | BAI KAMARA JR.                          BAI KAMARA JR
da17cf2a81   | Genius         372053                                    4   8    | WELLE: ERDBALL                          WELLE:ERDBALL
69c2f5bdfb   | Discogs        237971                                    4   8    | FLOWIN IMMO                             FLOWIN' IMMO
9764682b13   | Genius         21890                                     4   8    | FLOWIN IMMO                             FLOWINIMMO
4d183ffd47   | Genius         1196956                                   4   5    | MACHINECODE                             MACHINE CODE
d8ed905cd2   | MusicBrainz    68693707807897858389324918845492291057    4   8    | MACHINECODE                             MACHINE CODE
f91b1c7cc6   | Traxsource     250514                                    4   8    | MACHINECODE                             MACHINE CODE
93d2850288   | Discogs        405807                                    4   8    | JACOBS DREAM                            JACOB'S DREAM
28b4869fb6   | Genius         359515                                    4   7    | JACOBS DREAM                            JACOB'S DREAM
""")
pdbm.master()
pdbm.merge()

In [ ]:
pdbm = PanDBMatch(baseDB, verbose=True)
pdbm.select(minQual=4, maxQual=5, minName=4)

In [ ]:
pdbm.include("""
717fb648a3   | Traxsource     18324                                     5   4    | MARK VERBOS                             MARK VERBOS
""")
pdbm.master()
pdbm.merge()

In [ ]:
pdbm = PanDBMatch(baseDB, verbose=True)
pdbm.select(minQual=2, maxQual=4, minName=3)

In [ ]:
pdbm.include("""
bb32f67cbe   | Genius         453859                                    5   3    | EDDY WALLY                              EDDY WALLY
36d51199ea   | Spotify        68jFQKiOk63vHQSTub3oCQ                    5   2    | CELTIC SPIRIT                           CELTIC SPIRIT
""")
pdbm.master()
pdbm.merge()

In [ ]:
pdbm = PanDBMatch(baseDB, verbose=True)
pdbm.select(minQual=1, maxQual=2, minName=5)

In [ ]:
pdbm.include("""
901a18ca51   | Genius         392633                                    5   1    | SUNSET SONS                             SUNSET SONS
db0c2876ba   | Genius         384307                                    5   1    | RENZO ARBORE                            RENZO ARBORE
071de29638   | Beatport       214771                                    5   1    | KLUBBINGMAN                             KLUBBINGMAN
24d200e23d   | Genius         1019766                                   5   1    | HERMAN FRANK                            HERMAN FRANK
658ed54101   | Discogs        1623172                                   5   1    | KING CLAVE                              KING CLAVE
f11385d230   | Beatport       398443                                    5   1    | ENZO AVITABILE                          ENZO AVITABILE
00aa7f6412   | Genius         1139187                                   5   1    | MARK ORTON                              MARK ORTON
c477be4fd0   | Genius         455950                                    5   1    | SAM HULICK                              SAM HULICK
""")

pdbm.master()
pdbm.merge()

In [ ]:
del pdbm

In [ ]:
# Fix This:
#afc404588f   | 183166                   Discogs        113700                                  2    STEVE MASTERSON                                   STEVE MAESTRO                                      | afc404588f
#a6da8c9ee9   | 25875                    Discogs        678236                                  4    ALTAR OF FLESH                                    ALTAR OF FLIES                                     | a6da8c9ee9

# New Matching Code

# New Single Matching Code

In [ ]:
names = smdb.baseIO.getAvailableNames()

In [ ]:
smdb = SingleMatchDB(baseDB="RateYourMusic", compareDB="Spotify", reqs=reqs)
smdb.match()
smdb.save()
del smdb


mdbpd = MusicDBPermDir()
mres  = io.get(mdbpd.getMatchPermPath().join("primaryMatch.p"))
scmdb = SingleCrossMatchDB(baseDB, mres, crossreqs, verbose=True)
scmdb.match()
scmdb.save()
del scmdb


In [ ]:
pdbio = PanDBIO()
mmeDF = pdbio.getData()

In [ ]:
mmeDF[mmeDF["RateYourMusic"] == '106836']

In [ ]:
mmeDF[mmeDF["Spotify"] == '3lk3F4u5qqxq8zFjwNj5U1']

In [ ]:
pdbm = PanDBMatch(baseDB, verbose=True)
pdbm.select(minQual=5)
pdbm.masterSingle()
#pdbm.merge(allownew=False, verbose=False)

In [ ]:
pdbm = PanDBMatch(baseDB, verbose=True)
pdbm.select(minQual=4, maxQual=5)

In [ ]:
pdbm.include("""
1d2402a17a   | 1061693                  Spotify        49D8h67pxvvUNGOLKEGjkx                  4    OWAIN ARWEL HUGHES                                OWAIN ARWEL HUGHES                                 | 1d2402a17a
a86b2ef789   | 121809                   Spotify        6NSIW1uEq8JZmxEkHMF17c                  4    ANNA TOMOWA-SINTOW                                ANNA TOMOWA-SINTOW                                 | a86b2ef789
877d262f5e   | 142182                   Spotify        5DwQvVHPVspRvStEAN722N                  4    TAKÁCS QUARTET                                   TAKÁCS QUARTET                                    | 877d262f5e
a2f65f8447   | 412578                   Spotify        50skve7Y0al39yGqLuCMNu                  4    MAURICE ABRAVANEL                                 MAURICE ABRAVANEL                                  | a2f65f8447
""")
pdbm.masterSingle()
pdbm.merge(allownew=False, verbose=False)

In [ ]:
pdbm = PanDBMatch(baseDB, verbose=True)
pdbm.select(minQual=2, maxQual=4)

In [ ]:
pdbm.include("""
9554709d72   | 405351                   Spotify        2mHCS8PPaV7cAmT3ew8qHY                  2    SAULIUS SONDECKIS                                 SAULIUS SONDECKIS                                  | 9554709d72
""")
pdbm.masterSingle()
pdbm.merge(allownew=False, verbose=False)

In [ ]:
pdbm = PanDBMatch(baseDB, verbose=True)
pdbm.select(minQual=1, maxQual=2)

In [ ]:
pdbm.include("""
3178f6847b   | 337551                   Spotify        7N0fh2csz0eFkrE01LF1m3                  1    STRATOS PAGIOUMTZIS                               STRATOS PAGIOUMTZIS                                | 3178f6847b
842d333cee   | 375588                   Spotify        2LqWWIvCBaetjLStxk1XK6                  1    VAN AND SCHENCK                                   VAN & SCHENCK                                      | 842d333cee
60cc9bc61a   | 77193                    Spotify        6VeTIJi6Dlx8ywPfIwqALY                  1    ALBERT NICHOLAS                                   ALBERT NICHOLAS                                    | 60cc9bc61a
""")
pdbm.masterSingle()
pdbm.merge(allownew=False, verbose=False)